# 01 TripoSR 单图→Mesh（最终加固）

In [ ]:
import sys, subprocess, types
from pathlib import Path
from PIL import Image
Image.MAX_IMAGE_PIXELS = 40_000_000
import torch
assert torch.cuda.is_available()
print('GPU', torch.cuda.get_device_name(0))
WORK=Path('/kaggle/working'); OUT=WORK/'outputs_triposr'; IN=WORK/'inputs'
OUT.mkdir(exist_ok=True); IN.mkdir(exist_ok=True)

def pip(*a):
    r=subprocess.run([sys.executable,'-m','pip','install','-q',*a], capture_output=True, text=True)
    print('pip', a, r.returncode)
    if r.returncode: print((r.stderr or '')[-1000:])

pip('einops','omegaconf','Pillow','huggingface_hub','trimesh','opencv-python-headless','imageio','scikit-image','matplotlib','xatlas')
pip('git+https://github.com/tatsy/torchmcubes.git')

REPO=WORK/'TripoSR'
if not REPO.exists():
    subprocess.run(f'git clone --depth 1 https://github.com/VAST-AI-Research/TripoSR.git {REPO}', shell=True, check=True)

# rembg stub + patch run.py
sys.modules['rembg'] = types.SimpleNamespace(remove=lambda img, *a, **k: img)
run_py=REPO/'run.py'
if run_py.exists():
    txt=run_py.read_text(encoding='utf-8')
    if 'import rembg' in txt and 'SimpleNamespace' not in txt:
        txt=txt.replace('import rembg', 'import types as _types; rembg=_types.SimpleNamespace(remove=lambda x,*a,**k:x)')
        run_py.write_text(txt, encoding='utf-8')
print('ready')

In [ ]:
from pathlib import Path
from PIL import Image
import urllib.request
Image.MAX_IMAGE_PIXELS = 40_000_000
IN=Path('/kaggle/working/inputs')

def ok(p):
    try:
        if p.stat().st_size > 12*1024*1024: return None
        im=Image.open(p).convert('RGB')
        if im.size[0]*im.size[1] > 8_000_000: return None
        return im
    except Exception:
        return None

cands=list(Path('/kaggle/working').rglob('hero_000.png'))
im=None; chosen=None
for p in cands:
    im=ok(p)
    if im is not None:
        chosen=p; break
if im is None:
    demo=IN/'chair.png'
    urllib.request.urlretrieve('https://raw.githubusercontent.com/VAST-AI-Research/TripoSR/main/examples/chair.png', demo)
    im=Image.open(demo).convert('RGB'); chosen=demo
im.thumbnail((1024,1024)); inp=IN/'input.png'; im.save(inp)
print('using', chosen, im.size)
try: display(im.resize((280,280)))
except Exception: pass

In [ ]:
import sys, subprocess, traceback, shutil, types, inspect
from pathlib import Path
from PIL import Image as PILImage
import torch

REPO=Path('/kaggle/working/TripoSR'); OUT=Path('/kaggle/working/outputs_triposr'); inp=Path('/kaggle/working/inputs/input.png')
sys.path.insert(0,str(REPO))
sys.modules['rembg'] = types.SimpleNamespace(remove=lambda img, *a, **k: img)
errors=[]; arts=[]

r=subprocess.run(f'cd {REPO} && python run.py "{inp}" --output-dir "{OUT}" --model-save-format obj', shell=True, capture_output=True, text=True)
print('CLI rc', r.returncode)
print((r.stdout or '')[-2000:])
if r.returncode!=0:
    print((r.stderr or '')[-2000:]); errors.append(r.stderr or '')
arts=list(OUT.rglob('*.obj'))+list(OUT.rglob('*.glb'))+list(OUT.rglob('*.ply'))
print('CLI arts', arts)

if not arts:
    try:
        from tsr.system import TSR
        model=TSR.from_pretrained('stabilityai/TripoSR', config_name='config.yaml', weight_name='model.ckpt')
        model.renderer.set_chunk_size(8192); model.to('cuda')
        image=PILImage.open(inp).convert('RGB')
        with torch.no_grad():
            codes=model([image], device='cuda')
            # 兼容不同版本签名
            sig=inspect.signature(model.extract_mesh)
            print('extract_mesh sig', sig)
            kwargs={}
            params=sig.parameters
            if 'has_vertex_color' in params:
                kwargs['has_vertex_color'] = True
            if 'resolution' in params:
                kwargs['resolution'] = 256
            # positional scene_codes first
            try:
                meshes=model.extract_mesh(codes, **kwargs)
            except TypeError:
                # older: extract_mesh(scene_codes, resolution=)
                meshes=model.extract_mesh(codes, True, 256)
        outp=OUT/'mesh.obj'
        meshes[0].export(str(outp))
        arts=[outp]
        print('API ok', outp)
    except Exception:
        tb=traceback.format_exc(); errors.append(tb); print(tb)

print('final arts', arts)
try:
    import numpy as np
    import matplotlib; matplotlib.use('Agg')
    import matplotlib.pyplot as plt, trimesh
    if arts:
        mesh=trimesh.load(str(arts[0]), force='mesh'); v=np.asarray(mesh.vertices)
        fig=plt.figure(figsize=(5,5)); ax=fig.add_subplot(111, projection='3d')
        step=max(len(v)//4000,1); ax.scatter(v[::step,0],v[::step,1],v[::step,2],s=1)
        fig.savefig(OUT/'preview.png'); plt.close(fig)
except Exception as e:
    print('preview skip', e)
shutil.make_archive('/kaggle/working/triposr_export','zip', OUT)
assert arts, 'no mesh '+str(errors)[:800]
print('01 DONE', arts)